# Transform Circuits Data

1. Read bronze circuits table
2. Keep only the columns required for Analytics (drop url column)
3. Standardise column names using snake_case
4. Rename columns to make them more meaningful
5. Filter out rows where circuit_id is null
6. Remove duplicated values
7. Transform values of columns in Title Case
8. Write the transformted data to a silver table 

In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

## Step 1 - Read bronze `circuits` table

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_df = spark.read.table(bronze_table).filter((F.col('batch_id') == v_batch_id))


In [0]:
display(circuits_df)

## Step 2 - Keep only the columns required for Analytics (drop url column)


In [0]:
circuits_df_selected = circuits_df.select(
    F.col('circuitId'),
    F.col('circuitName'),
    F.col('lat'),
    F.col('long'),
    F.col('locality'),
    F.col('country'),
    F.col('ingestion_timestamp'),
    F.col('source_file'),
    F.col('batch_id')
)

## Step 3 and 4 - Rename columns names


In [0]:
circuits_df_renamed = circuits_df_selected.withColumnsRenamed(
    {
        'circuitId': 'circuit_id',
        'circuitName': 'circuit_name',
        'lat': 'latitude',
        'long': 'longitude'
    }
)

## Step 5 - Filter out rows where circuit_id is null

In [0]:
circuits_not_null = circuits_df_renamed.filter(F.col('circuit_id').isNotNull())

In [0]:
display(circuits_not_null)

## Step 6 - Remove duplicated values

In [0]:
circuits_distinct_df = circuits_not_null.distinct()

In [0]:
display(circuits_distinct_df)

## Step 7 - Transform values of columns in Title Case

In [0]:
circuits_df_final = circuits_distinct_df.withColumns(
    {
        'circuit_name': F.initcap('circuit_name'),
        'locality': F.initcap('locality')

    }
)

## Final Step - Write the transformted data to a silver table 

In [0]:
write_to_silver(
    df = circuits_df_final,
    target_table = silver_table,
    table_key = 't.circuit_id == s.circuit_id',
    columns_to_update = [
        'circuit_name', 
        'latitude', 
        'longitude', 
        'locality', 
        'country', 
        'ingestion_timestamp',
        'source_file',
        'batch_id'
    ]
)

In [0]:
# circuits_df_final = (
#     circuits_df_final
#         .withColumn('created_timestamp', F.current_timestamp())
#         .withColumn('updated_timestamp', F.current_timestamp())
# )

In [0]:
# if not spark.catalog.tableExists(silver_table):
#     (
#         circuits_df_final.write
#             .format('delta')
#             .mode('overwrite')
#             .saveAsTable(silver_table)
#     )
# else:
#     from delta.tables import DeltaTable

#     delta_table = DeltaTable.forName(spark, silver_table)
#     (
#         delta_table.alias('t')
#         .merge (
#             circuits_df_final.alias('s'),
#             't.circuit_id = s.circuit_id'
#         )
#         .whenMatchedUpdate(
#             condition='s.batch_id >= t.batch_id',
#             set = {
#                 'circuit_name': 's.circuit_name',
#                 'latitude': 's.latitude',
#                 'longitude': 's.longitude',
#                 'locality': 's.locality',
#                 'country': 's.country',
#                 'ingestion_timestamp': 's.ingestion_timestamp',
#                 'source_file': 's.source_file',
#                 'batch_id': 's.batch_id',
#                 'updated_timestamp': 's.updated_timestamp',
#             }
#         )
#         .whenNotMatchedInsertAll()
#     )



In [0]:
%sql
SELECT * FROM formula1.silver.circuits